# Case 2: Comparison to MD & MARLOWE results for Fe

This is a validation benchmark for comparison to data from 
> C. J. Ortiz, L. Luneville, and D. Simeone, “Binary Collision Approximation,” > in Comprehensive Nuclear Materials, Elsevier, 2020, pp. 595–619

Fig. 9 of the above paper shows number of Frenkel pairs in Fe calculated with  MD and BCA(MARLOWE) for different capture radii.

Cascade simulations are performed with OpenTRIM with the following settings
- electronic stopping switched off 
- Ed = 40 eV
- Ec = 10 eV
- PKA energy and Rc as in the publication  

### 1. Import libs, set Model Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import opentrim
print('opentrim', opentrim.__version__)

a0 = 0.28665    # Fe lattice const [nm]
rho_fe = 7.8658 # Fe mass density
L = 1000.0     # simulation box size [nm]
Ed = 40.0       # Fe Displacement enrgy [eV]
Ec = 10.0       # Ion energy lower cut-off [eV]
Rc = np.array([2, 2.8, 4])*a0  # Fe defect recombination radius 
E0 = np.array([5, 10, 20, 40, 80])*1000    # Initial PKA energy [eV]
Nh = 1000      # Histories to run

### 2. Setup OpenTRIM Config

In [ ]:
config = opentrim.Config()

config.Simulation.simulation_type = opentrim.SimulationType.CascadesOnly
config.Simulation.electronic_stopping = opentrim.Stopping.Off
config.Simulation.defect_recombination = True

config.Transport.flight_path_type = opentrim.FlightPath.Variable
config.Transport.min_energy = Ec

config.IonBeam.ion = opentrim.Element('Fe')
config.IonBeam.spatial_distribution.geometry = opentrim.Geometry.Volume
config.IonBeam.spatial_distribution.center = np.array([0.5, 0.5, 0.5])*L

config.Target.size = np.array([1,1,1])*L
config.Target.periodic_bc = [0,0,0]

# material
mat = opentrim.Material(); mat.id = 'Fe'; mat.density = rho_fe
atom = opentrim.Atom(); atom.element = opentrim.Element('Fe')
atom.X = 1.0; atom.Ed = Ed; atom.El = 0.001; atom.Er = Ed; 
mat.composition.append(atom)
config.Target.materials.append(mat)
#region
region = opentrim.Region(); region.id = 'cube'; region.material_id = 'Fe'
region.size = np.array([1,1,1])*L
config.Target.regions.append(region)

# Run
config.Run.max_no_ions = Nh

config.validate() # check it
config # show it

### 3. Run all cases

In [ ]:
# array to recieve results
Nv = np.zeros((len(E0),len(Rc)))
Nt = len(E0)*len(Rc);

# run all cases in series
for i in range(len(E0)):
    for j in range(len(Rc)):
        # prepare config
        config.IonBeam.energy_distribution.center = E0[i]
        config.Target.materials[0].composition[0].Rc = Rc[j]
        # get a simulation driver and run it
        t0 = time.perf_counter()
        D = opentrim.Driver(config)
        D.run()
        D.wait() # wait for results
        t1 = time.perf_counter()
        # get the total tally
        info = opentrim.Info(D)
        x, dx = info['tally']['totals']['data']
        Nv[i,j] = x[1,1]
        print(f"{i*len(Rc) + j + 1:d}/{Nt:d}.","E[keV]=",E0[i]/1000,"Rc[nm]=",Rc[j],f"time[s]={t1-t0:.4f}")

### 4. Plot the results


In [ ]:
plt.figure(figsize=(6,4.5),dpi=80)
plt.plot(E0/1000, Nv, "o-", label=["Rc=2a0","Rc=2.8a0","Rc=4a0"])
plt.xlabel('PKA energy (keV)')
plt.ylabel('# of Frenkel Pairs')
plt.xlim(0,100)
plt.ylim(0,500)
plt.title(f'OpenTRIM Cascade damage in Fe')
plt.legend()
plt.tight_layout()
plt.show()